# Download NISAR GCOV granules into Google Drive

Internal-convenience mirror of the raw granules. The repo manifest
(`data/granule_manifest.csv`) stays the source of truth; this just saves a
re-download for teammates. **Does not change any storage policy or docs.**

Run the cells top to bottom. The download loop is **resumable** - granules
already in Drive at full size are skipped.

> Free Google Drive is 15 GB total. A GCOV granule is ~1.5-2 GB, so the 4
> dual-pol granules are ~8 GB and all 8 are ~16 GB. On a free account use
> `SCOPE = "dualpol"`.

## 1. Install dependencies

In [ ]:
!pip -q install asf_search pandas

## 2. Mount Google Drive and log in to NASA Earthdata

Earthdata Login is required even though NISAR GCOV is public. This builds an
**`asf_search` session** and hands it to every download, so the credentials
actually reach the data host (the earlier `HTTP 401` happened because the
download used an unauthenticated session). Enter your Earthdata username and
password at the prompts.

> If you still get `401` after a correct username/password, your Earthdata
> account most likely needs to **accept the ASF data EULA / authorise the
> "Alaska Satellite Facility Data Access" application** once at
> <https://urs.earthdata.nasa.gov/profile> -> Applications -> Authorized Apps.

In [ ]:
from getpass import getpass

from google.colab import drive

drive.mount("/content/drive")

import asf_search as asf

# Build an asf_search session and authenticate it with Earthdata Login creds.
# This session is passed into every .download() below so auth reaches the
# data host (fixes the HTTP 401 seen with an unauthenticated download).
edl_user = input("Earthdata username: ").strip()
edl_pass = getpass("Earthdata password: ")
asf_session = asf.ASFSession().auth_with_creds(edl_user, edl_pass)
print("asf_search session authenticated for", edl_user)

## 3. Configuration

Edit `DRIVE_DIR` and `SCOPE` as needed. `SCOPE = "dualpol"` pulls only the
HH+HV granules (the 4 usable for the main experiment); `"all"` pulls all 8.

In [ ]:
# Permanent destination in YOUR Drive. Pick a folder you can share with the team.
DRIVE_DIR = "/content/drive/MyDrive/nisar-flood/raw_granules"

# Read the locked manifest straight from the repo so IDs match what the team agreed on.
MANIFEST_URL = (
    "https://raw.githubusercontent.com/cs59300-deep-learning/"
    "nisar-flood-mapping/main/data/granule_manifest.csv"
)

# "dualpol" -> only HH+HV granules (4); "all" -> every granule (8).
SCOPE = "dualpol"

# Treat anything smaller than this as a partial download and re-fetch it.
MIN_VALID_BYTES = 500 * 1024 * 1024  # 500 MB

## 4. Download loop

Downloads each selected granule from NASA ASF directly into `DRIVE_DIR`,
skipping any already present, and prints a summary at the end.

In [ ]:
import os

import pandas as pd

# asf_search was imported and 'asf_session' created in cell 2.
os.makedirs(DRIVE_DIR, exist_ok=True)

manifest = pd.read_csv(MANIFEST_URL)
print(f"Manifest: {len(manifest)} granules")

# Dual-pol rows carry HV; single-pol (HH-only) rows do not.
if SCOPE == "dualpol":
    pol = manifest["polarisations"].str.upper().str.replace(" ", "")
    selected = manifest[pol.str.contains("HV")]
elif SCOPE == "all":
    selected = manifest
else:
    raise ValueError(f"SCOPE must be 'dualpol' or 'all', got {SCOPE!r}")

granule_ids = selected["granule_id"].tolist()
print(f"Scope '{SCOPE}': {len(granule_ids)} granule(s) to ensure present\n")

EXT = ".h5"  # GCOV product is an HDF5 file


def already_have(granule_id: str) -> bool:
    path = os.path.join(DRIVE_DIR, granule_id + EXT)
    return os.path.exists(path) and os.path.getsize(path) >= MIN_VALID_BYTES


downloaded, skipped, failed = [], [], []

for i, gid in enumerate(granule_ids, 1):
    print(f"[{i}/{len(granule_ids)}] {gid}")
    if already_have(gid):
        print("    already in Drive (full size) - skipping")
        skipped.append(gid)
        continue
    try:
        results = asf.granule_search([gid])
        if len(results) == 0:
            results = asf.granule_search([gid + "-GCOV"])  # fallback form
        if len(results) == 0:
            print("    NOT FOUND on ASF - check the granule_id")
            failed.append(gid)
            continue
        results[0].download(path=DRIVE_DIR, session=asf_session)
        got = sorted(
            (f for f in os.listdir(DRIVE_DIR) if gid in f and f.endswith(EXT)),
            key=lambda f: os.path.getsize(os.path.join(DRIVE_DIR, f)),
            reverse=True,
        )
        if got:
            src = os.path.join(DRIVE_DIR, got[0])
            dst = os.path.join(DRIVE_DIR, gid + EXT)
            if src != dst:
                os.replace(src, dst)
            size_gb = os.path.getsize(dst) / 1e9
            print(f"    saved {gid}{EXT}  ({size_gb:.2f} GB)")
            downloaded.append(gid)
        else:
            print("    download produced no .h5 - check ASF product type")
            failed.append(gid)
    except Exception as exc:
        msg = str(exc)
        print(f"    ERROR: {msg}")
        if "401" in msg:
            print("    -> 401 means auth did not reach the data host. Re-run")
            print("       cell 2, and authorise 'Alaska Satellite Facility")
            print("       Data Access' at urs.earthdata.nasa.gov/profile.")
        failed.append(gid)

print("\n==== summary ====")
print(f"downloaded : {len(downloaded)}")
print(f"skipped    : {len(skipped)} (already present)")
print(f"failed     : {len(failed)}")
if failed:
    print("failed IDs:")
    for gid in failed:
        print("  ", gid)
print(f"\nGranules live in: {DRIVE_DIR}")
print("Share THAT Drive folder with your teammates (view/download access).")